<a href="https://colab.research.google.com/github/Gagangowdatg/BIS_1BM24CS098/blob/main/Bis_1bm24cs098.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random

# --- CONFIGURATION & HYPERPARAMETERS ---
STAFF = ['Dr. Alice', 'Dr. Bob', 'Nurse Charlie', 'Nurse David', 'Nurse Eve']
DAYS = 7
SHIFTS = ['D', 'N', 'O'] # D = Day Shift, N = Night Shift, O = Off (Rest Day)

POP_SIZE = 50
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.15
MAX_GEN = 200

# Hospital Rules / Constraints
REQUIRED_STAFF_PER_SHIFT = 2 # Min staff needed per Day/Night shift for patient demand
MAX_SHIFTS_PER_WEEK = 5 # Legal limit of shifts per person per week

# --- CORE GA FUNCTIONS ---

def create_individual():
    """Generates a random weekly schedule matrix for the hospital staff."""
    schedule = {}
    for person in STAFF:
        schedule[person] = [random.choice(SHIFTS) for _ in range(DAYS)]
    return schedule

def calculate_fitness(schedule):
    """
    Evaluates the schedule quality based on structural hospital constraints.
    Higher fitness values correspond to fewer schedule violations.
    """
    penalty = 0

    # 1. Check Shift Sufficiency (Patient Demand)
    for day in range(DAYS):
        for shift in ['D', 'N']:
            assigned_count = sum(1 for person in STAFF if schedule[person][day] == shift)
            if assigned_count < REQUIRED_STAFF_PER_SHIFT:
                # Critical understaffing penalty
                penalty += 100 * (REQUIRED_STAFF_PER_SHIFT - assigned_count)

    # 2. Check Labor Law & Fatigue Rules
    for person in STAFF:
        shifts = schedule[person]
        total_work_shifts = sum(1 for s in shifts if s in ['D', 'N'])

        # Overtime Penalty
        if total_work_shifts > MAX_SHIFTS_PER_WEEK:
            penalty += 50 * (total_work_shifts - MAX_SHIFTS_PER_WEEK)

        # Fatigue Penalty: Night shift immediately followed by Day shift the next morning
        for day in range(DAYS - 1):
            if shifts[day] == 'N' and shifts[day+1] == 'D':
                penalty += 75

    # Convert minimization problem to maximization problem. Avoid division by zero.
    return 1 / (1 + penalty)

def select_parent(population, fitnesses):
    """Selects one parent using Tournament Selection (Size = 3)."""
    tournament = random.sample(list(zip(population, fitnesses)), 3)
    tournament.sort(key=lambda x: x[1], reverse=True)
    return tournament[0][0]

def crossover(parent1, parent2):
    """Combines two schedules by splitting them at a random day boundary."""
    child1, child2 = {}, {}
    crossover_point = random.randint(1, DAYS - 1)

    for person in STAFF:
        child1[person] = parent1[person][:crossover_point] + parent2[person][crossover_point:]
        child2[person] = parent2[person][:crossover_point] + parent1[person][crossover_point:]

    return child1, child2

def mutate(individual):
    """Randomly flips individual shifts to discover better allocation mixes."""
    for person in STAFF:
        for day in range(DAYS):
            if random.random() < MUTATION_RATE:
                individual[person][day] = random.choice(SHIFTS)
    return individual

# --- MAIN EXECUTION PIPELINE ---

def run_genetic_algorithm():
    # Step 2: Initialize population
    population = [create_individual() for _ in range(POP_SIZE)]
    best_solution = None
    best_fitness = -1

    print("Evolution starting...")

    # Step 3: Main generational loop
    for gen in range(1, MAX_GEN + 1):
        # Step 4: Evaluate fitness
        fitnesses = [calculate_fitness(ind) for ind in population]

        # Step 5: Update the historically best solution
        for i, fit in enumerate(fitnesses):
            if fit > best_fitness:
                best_fitness = fit
                best_solution = pickle_copy(population[i]) # Deep copy equivalent

        # Optional optimization print log
        if gen % 50 == 0 or gen == 1:
            raw_penalty = int((1 / best_fitness) - 1) if best_fitness > 0 else "Inf"
            print(f"Generation {gen:3d} | Highest Fitness: {best_fitness:.5f} (Violations Penalty Score: {raw_penalty})")

        # Step 6: Empty tracking pool
        new_population = []

        # Step 7: Breeding loop
        while len(new_population) < POP_SIZE:
            # Step 8: Selection
            p1 = select_parent(population, fitnesses)
            p2 = select_parent(population, fitnesses)

            # Steps 9-13: Crossover application
            if random.random() < CROSSOVER_RATE:
                c1, c2 = crossover(p1, p2)
            else:
                c1, c2 = pickle_copy(p1), pickle_copy(p2)

            # Step 14: Mutation execution
            c1 = mutate(c1)
            c2 = mutate(c2)

            # Step 15: Population accumulation
            new_population.append(c1)
            if len(new_population) < POP_SIZE:
                new_population.append(c2)

        # Step 17: Advance generation
        population = new_population

    return best_solution, best_fitness

def pickle_copy(obj):
    """Quick deep-copy helper function for dictionary structures."""
    import pickle
    return pickle.loads(pickle.dumps(obj))

# Run the schedule search simulation
if __name__ == "__main__":
    best_schedule, final_fitness = run_genetic_algorithm()
    final_penalty = int((1 / final_fitness) - 1)

    print("\n" + "="*45)
    print(f"FINAL OPTIMIZATION REPORT (Penalty Score: {final_penalty})")
    print("="*45)

    # Print clean readable table rows for roster mapping
    days_header = " | ".join([f"Day {i+1}" for i in range(DAYS)])
    print(f"{'Staff Member':<15} | {days_header}")
    print("-" * 70)
    for staff_member, shift_list in best_schedule.items():
        shifts_row = " | ".join(shift_list)
        print(f"{staff_member:<15} | {shifts_row}")
    print("="*45)


Evolution starting...
Generation   1 | Highest Fitness: 0.00129 (Violations Penalty Score: 775)
Generation  50 | Highest Fitness: 0.00249 (Violations Penalty Score: 400)
Generation 100 | Highest Fitness: 0.00285 (Violations Penalty Score: 350)
Generation 150 | Highest Fitness: 0.00307 (Violations Penalty Score: 325)
Generation 200 | Highest Fitness: 0.00307 (Violations Penalty Score: 325)

FINAL OPTIMIZATION REPORT (Penalty Score: 325)
Staff Member    | Day 1 | Day 2 | Day 3 | Day 4 | Day 5 | Day 6 | Day 7
----------------------------------------------------------------------
Dr. Alice       | N | D | D | O | D | D | D
Dr. Bob         | O | N | N | N | N | N | O
Nurse Charlie   | N | O | D | D | D | D | D
Nurse David     | D | N | N | O | N | N | N
Nurse Eve       | D | D | O | N | O | O | N
